# PostgreSQL Data Load — Verification Notebook
Connects to the PostgreSQL database and verifies all 5 tables loaded correctly after migration.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import warnings
warnings.filterwarnings('ignore')

DB_USER     = 'postgres'
DB_PASSWORD = 'CIpass2024@@'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'carinsurance_db'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('Connected to PostgreSQL successfully!')

Connected to PostgreSQL successfully!


## 1. Table Row Counts

In [2]:
tables = ['policyholders', 'vehicles', 'policies', 'claims', 'payments']
print(f'{"Table":20s} {"Row Count":>10s}')
print('-' * 32)
for t in tables:
    with engine.connect() as conn:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
    print(f'{t:20s} {count:>10,}')

Table                 Row Count
--------------------------------
policyholders             2,894
vehicles                  2,894
policies                  2,894
claims                    1,927
payments                  3,376


## 2. Sample Data — Policyholders

In [3]:
df = pd.read_sql('SELECT * FROM policyholders LIMIT 5', engine)
df

,policyholder_id,full_name,date_of_birth,age,gender,state,phone,email,license_years,prior_claims_count,credit_score,marital_status
0,PH00001,Kevin Miller,1991-03-30,33,Male,WI,9782452545.0,yuki205@hotmail.com,25,1,784,Unknown
1,PH00002,Kevin Miller,1988-07-20,36,Female,AZ,5462391930.0,john372@gmail.com,31,0,676,Unknown
2,PH00003,Maria Anderson,1996-07-05,28,Male,PA,9658410741.0,yuki701@hotmail.com,33,2,539,Single
3,PH00004,Michael Moore,1962-09-19,62,Male,IL,7357800313.0,priya964@hotmail.com,4,0,725,Unknown
4,PH00005,NINA JACKSON,1968-06-30,56,Male,TN,None,lisa867@outlook.com,10,4,625,Married


## 3. Sample Data — Claims

In [4]:
df = pd.read_sql('SELECT * FROM claims LIMIT 5', engine)
df

,claim_id,policy_id,claim_date,claim_type,claim_amount_usd,settled_amount_usd,status,description,is_fraud_flag,days_to_report
0,CLM00001,POL00966,2023-09-09,Natural Disaster,28554.49,0.0,Rejected,Hit and run incident,False,18
1,CLM00002,POL01661,2024-03-01,Vandalism,52455.85,0.0,Rejected,Front end collision on highway,False,46
2,CLM00003,POL01761,2021-05-02,Fire,78288.69,53347.9,Settled,Windshield cracked by debris,False,39
3,CLM00004,POL02123,2020-01-18,Theft,60430.71,0.0,Pending,Vandalism - scratches and dents,False,53
4,CLM00005,POL00616,2022-08-03,Theft,31226.82,0.0,Approved,Windshield cracked by debris,False,1


## 4. Migration Health Check

In [5]:
query = '''
    SELECT
        (SELECT COUNT(*) FROM policyholders)                             AS total_policyholders,
        (SELECT COUNT(*) FROM policies)                                  AS total_policies,
        (SELECT COUNT(*) FROM claims)                                    AS total_claims,
        (SELECT COUNT(*) FROM claims WHERE is_fraud_flag = TRUE)         AS fraud_claims,
        (SELECT COUNT(*) FROM payments WHERE is_late = TRUE)             AS late_payments,
        (SELECT ROUND(AVG(premium_usd)::numeric,2) FROM policies)        AS avg_premium,
        (SELECT ROUND(AVG(credit_score)::numeric,1) FROM policyholders)  AS avg_credit_score
'''
result = pd.read_sql(query, engine)
result.T.rename(columns={0: 'Value'})

,Value
total_policyholders,2894.00
total_policies,2894.00
total_claims,1927.00
fraud_claims,155.00
late_payments,514.00
avg_premium,1882.66
avg_credit_score,680.00


## 5. High Risk Policyholders by State

In [6]:
query = '''
    SELECT state,
           COUNT(*) AS high_risk_count,
           ROUND(AVG(credit_score)::numeric, 1) AS avg_credit_score
    FROM policyholders
    WHERE credit_score < 550 OR prior_claims_count >= 5
    GROUP BY state
    ORDER BY high_risk_count DESC
    LIMIT 10
'''
df = pd.read_sql(query, engine)
print('Top 10 States by High-Risk Policyholders:')
df

Top 10 States by High-Risk Policyholders:


,state,high_risk_count,avg_credit_score
0,CA,11,512.2
1,GA,10,513.7
2,OH,10,517.4
3,NJ,10,516.7
4,NY,10,527.4
5,VA,10,509.0
6,MI,9,516.7
7,TX,9,501.7
8,WI,8,527.4
9,IL,8,509.8


## 6. Fraud Analysis by Claim Type

In [7]:
query = '''
    SELECT claim_type,
           COUNT(*) AS total,
           SUM(CASE WHEN is_fraud_flag THEN 1 ELSE 0 END) AS fraud_count,
           ROUND(AVG(claim_amount_usd)::numeric, 2) AS avg_amount
    FROM claims
    GROUP BY claim_type
    ORDER BY fraud_count DESC
'''
df = pd.read_sql(query, engine)
df

,claim_type,total,fraud_count,avg_amount
0,Theft,320,35,40848.37
1,Flood,345,28,39703.29
2,Fire,307,27,37615.97
3,Natural Disaster,337,25,37639.90
4,Accident,307,21,37283.34
5,Vandalism,311,19,37377.91


## 7. Payment Method Breakdown

In [8]:
query = '''
    SELECT payment_method,
           COUNT(*) AS total_payments,
           SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_count,
           ROUND(AVG(amount_usd)::numeric, 2) AS avg_amount
    FROM payments
    GROUP BY payment_method
    ORDER BY total_payments DESC
'''
df = pd.read_sql(query, engine)
df

,payment_method,total_payments,late_count,avg_amount
0,Credit Card,706,106,2081.47
1,Bank Transfer,687,105,2051.16
2,Direct Debit,683,117,2159.25
3,Cheque,677,92,2034.33
4,Cash,623,94,2058.20
